In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from torch.optim import lr_scheduler


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import pandas as pd

#df = pd.read_csv('/data/home/anisha23/master-thesis/ETTh1.csv')
df = pd.read_csv('/data/home/megvij23/Thesis/ETTh1.csv')
print(df.shape)

(17420, 8)


In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

features = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']

df[features] = scaler.fit_transform(df[features])

In [5]:
def create_sequences(data, target, seq_len=24, horizon=5):
    """
    data: [N, num_features]
    target: [N] (1D time series)
    """
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i:i+seq_len])             # input window
        y.append(target[i+seq_len:i+seq_len+horizon])  # next H steps
    return np.array(X), np.array(y)


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split

X, y = create_sequences(df[features].values, df['OT'])

print(X.shape)
print(y.shape)

(17392, 24, 6)
(17392, 5)


In [7]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Split train+val into train and val (e.g., 80% train, 20% val of original 80%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# Print shapes
print("X_train shape: ", X_train.shape)
print("X_val shape:   ", X_val.shape)
print("X_test shape:  ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_val shape: ", y_val.shape)
print("y_test shape: ", y_test.shape)

X_train shape:  (11130, 24, 6)
X_val shape:    (2783, 24, 6)
X_test shape:   (3479, 24, 6)
y_train shape:  (11130, 5)
y_val shape:  (2783, 5)
y_test shape:  (3479, 5)


In [8]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
horizon = 5
# ------------------------------
# 1. Fit scalers
# ------------------------------
input_scaler = StandardScaler()
target_scaler = StandardScaler()

# Scale inputs (fit on training set only)
X_train_scaled = input_scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val_scaled   = input_scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test_scaled  = input_scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

# Scale targets
y_train_scaled = target_scaler.fit_transform(y_train)   # shape: (n_samples, horizon)
y_val_scaled   = target_scaler.transform(y_val)
y_test_scaled  = target_scaler.transform(y_test)



# ------------------------------
# 2. Convert to tensors
# ------------------------------
X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_scaled = torch.tensor(y_train_scaled, dtype=torch.float32)

X_val = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_scaled = torch.tensor(y_val_scaled, dtype=torch.float32)

X_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_scaled = torch.tensor(y_test_scaled, dtype=torch.float32)

# ------------------------------
# 3. Create datasets & dataloaders
# ------------------------------
train_ds = TensorDataset(X_train, y_train_scaled)
val_ds   = TensorDataset(X_val, y_val_scaled)
test_ds  = TensorDataset(X_test, y_test_scaled)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64)
test_dl  = DataLoader(test_ds, batch_size=64)


In [9]:
import torch
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, horizon, num_layers=2, dropout=0.1):
        """
        Baseline LSTM for multi-step regression (no text embeddings).
        
        Args:
            input_dim (int): number of input features
            hidden_dim (int): hidden layer size
            horizon (int): number of forecast steps
            num_layers (int): number of LSTM layers
            dropout (float): dropout probability
        """
        super(LSTMRegressor, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, horizon)

    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, input_dim]
        Returns:
            preds: [batch, horizon]
        """
        _, (h_n, _) = self.lstm(x)            # h_n: [num_layers, batch, hidden_dim]
        h_last = h_n[-1]                      # take last layer's hidden state
        h_last = self.dropout(h_last)
        preds = self.fc(h_last)
        return preds


In [10]:
# Example dimensions
input_dim = 6       # number of features
hidden_dim = 64
horizon = 5

baseline_model = LSTMRegressor(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    horizon=horizon,
    num_layers=2,
    dropout=0.1
).to(device)


In [11]:
import torch
import numpy as np

def train_baseline_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, target_scaler, device):
    """
    Train and validate a baseline LSTM regression model (no prototypes, no text).

    Args:
        model: PyTorch model
        train_loader, val_loader: DataLoaders
        criterion: loss function (e.g., MSELoss)
        optimizer: torch optimizer
        num_epochs: number of epochs
        target_scaler: fitted StandardScaler for inverse transform
        device: 'cuda' or 'cpu'

    Returns:
        train_losses, val_losses, train_mse_orig, val_mse_orig
    """
    model.to(device)
    train_losses, val_losses = [], []
    train_mse_orig, val_mse_orig = [], []

    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        orig_preds_epoch, orig_targets_epoch = [], []

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device).float(), targets.to(device).float()

            optimizer.zero_grad()
            preds = model(inputs)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

            # Inverse-transform predictions and targets for real-scale metrics
            preds_np = preds.detach().cpu().numpy()
            targets_np = targets.detach().cpu().numpy()

            preds_orig = target_scaler.inverse_transform(preds_np)
            targets_orig = target_scaler.inverse_transform(targets_np)

            orig_preds_epoch.extend(preds_orig.reshape(-1))
            orig_targets_epoch.extend(targets_orig.reshape(-1))

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        train_mse_epoch_orig = np.mean((np.array(orig_preds_epoch) - np.array(orig_targets_epoch)) ** 2)
        train_mse_orig.append(train_mse_epoch_orig)

        # Validation
        model.eval()
        running_val_loss = 0.0
        orig_preds_val, orig_targets_val = [], []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device).float(), targets.to(device).float()
                preds = model(inputs)
                val_loss = criterion(preds, targets)
                running_val_loss += val_loss.item()

                preds_np = preds.detach().cpu().numpy()
                targets_np = targets.detach().cpu().numpy()
                preds_orig = target_scaler.inverse_transform(preds_np)
                targets_orig = target_scaler.inverse_transform(targets_np)

                orig_preds_val.extend(preds_orig.reshape(-1))
                orig_targets_val.extend(targets_orig.reshape(-1))

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        val_mse_epoch_orig = np.mean((np.array(orig_preds_val) - np.array(orig_targets_val)) ** 2)
        val_mse_orig.append(val_mse_epoch_orig)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"| Train Loss: {avg_train_loss:.6f} "
              f"| Val Loss: {avg_val_loss:.6f} "
              f"| Train MSE (orig): {train_mse_epoch_orig:.6f} "
              f"| Val MSE (orig): {val_mse_epoch_orig:.6f}")

    return train_losses, val_losses, train_mse_orig, val_mse_orig


In [12]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3, weight_decay=1e-5)

train_losses, val_losses, train_mse_orig, val_mse_orig = train_baseline_model(
    model=baseline_model,
    train_loader=train_dl,
    val_loader=val_dl,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=50,
    target_scaler=target_scaler,
    device=device
)


Epoch [1/50] | Train Loss: 0.619814 | Val Loss: 0.402231 | Train MSE (orig): 45.680912 | Val MSE (orig): 29.394766
Epoch [2/50] | Train Loss: 0.321833 | Val Loss: 0.273049 | Train MSE (orig): 23.721968 | Val MSE (orig): 19.962128
Epoch [3/50] | Train Loss: 0.275479 | Val Loss: 0.275644 | Train MSE (orig): 20.304100 | Val MSE (orig): 20.122593
Epoch [4/50] | Train Loss: 0.222000 | Val Loss: 0.195203 | Train MSE (orig): 16.364620 | Val MSE (orig): 14.418273
Epoch [5/50] | Train Loss: 0.190206 | Val Loss: 0.167701 | Train MSE (orig): 14.021358 | Val MSE (orig): 12.381310
Epoch [6/50] | Train Loss: 0.167030 | Val Loss: 0.144151 | Train MSE (orig): 12.303417 | Val MSE (orig): 10.683772
Epoch [7/50] | Train Loss: 0.152779 | Val Loss: 0.134556 | Train MSE (orig): 11.261972 | Val MSE (orig): 9.961702
Epoch [8/50] | Train Loss: 0.157045 | Val Loss: 0.142624 | Train MSE (orig): 11.576068 | Val MSE (orig): 10.557327
Epoch [9/50] | Train Loss: 0.139034 | Val Loss: 0.125605 | Train MSE (orig): 10.2

In [13]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_baseline(model, dataloader, target_scaler, device):
    """
    Evaluate the baseline LSTM model on a dataloader.

    Args:
        model: PyTorch model
        dataloader: DataLoader providing (inputs, targets)
        target_scaler: StandardScaler used to normalize targets
        device: 'cuda' or 'cpu'

    Returns:
        targets_all: np.array of true targets (original scale)
        preds_all: np.array of predictions (original scale)
    """
    model.eval()
    preds_all, targets_all = [], []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device).float(), targets.to(device).float()
            preds = model(inputs)

            # Move to CPU and inverse-transform
            preds_np = preds.detach().cpu().numpy()
            targets_np = targets.detach().cpu().numpy()

            preds_inv = target_scaler.inverse_transform(preds_np)
            targets_inv = target_scaler.inverse_transform(targets_np)

            preds_all.append(preds_inv)
            targets_all.append(targets_inv)

    preds_all = np.concatenate(preds_all, axis=0)
    targets_all = np.concatenate(targets_all, axis=0)

    return targets_all, preds_all


In [14]:
def compute_metrics(y_true, y_pred):
    """
    Compute regression performance metrics.
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2


In [15]:
# Evaluate on test set
targets_all, preds_all = evaluate_baseline(baseline_model, test_dl, target_scaler, device)

# Flatten for metrics (handles multi-horizon)
if preds_all.ndim > 1:
    preds_all = preds_all.reshape(-1)
    targets_all = targets_all.reshape(-1)

# Compute metrics
mse, rmse, mae, r2 = compute_metrics(targets_all, preds_all)

print("\n📈 Baseline Model Performance:")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")



📈 Baseline Model Performance:
MSE:  3.7952
RMSE: 1.9481
MAE:  1.3228
R²:   0.9477


In [16]:
import torch
import torch.nn as nn

class GRURegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, horizon, dropout=0.1, device=device):
        super(GRURegressor, self).__init__()
        self.device = device
        self.gru = nn.GRU(
            input_dim, hidden_dim, num_layers=2,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, horizon)
        self.to(self.device)

    def forward(self, x):
        x = x.to(self.device)
        _, h_n = self.gru(x)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)  # concat bi-directional
        h = self.dropout(h)
        out = self.fc(h)
        return out


In [17]:
model = GRURegressor(input_dim=6, hidden_dim=64, horizon=5, device=device).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train
train_losses, val_losses, *_ = train_baseline_model(
    model=model,
    train_loader=train_dl,
    val_loader=val_dl,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=50,
    target_scaler=target_scaler,
    device=device
)




Epoch [1/50] | Train Loss: 0.619545 | Val Loss: 0.358124 | Train MSE (orig): 45.664680 | Val MSE (orig): 26.346281
Epoch [2/50] | Train Loss: 0.275201 | Val Loss: 0.228774 | Train MSE (orig): 20.284130 | Val MSE (orig): 16.907629
Epoch [3/50] | Train Loss: 0.206497 | Val Loss: 0.217778 | Train MSE (orig): 15.214971 | Val MSE (orig): 16.149523
Epoch [4/50] | Train Loss: 0.171785 | Val Loss: 0.141794 | Train MSE (orig): 12.659892 | Val MSE (orig): 10.526014
Epoch [5/50] | Train Loss: 0.150551 | Val Loss: 0.133637 | Train MSE (orig): 11.098885 | Val MSE (orig): 9.924076
Epoch [6/50] | Train Loss: 0.133597 | Val Loss: 0.117855 | Train MSE (orig): 9.847697 | Val MSE (orig): 8.741462
Epoch [7/50] | Train Loss: 0.125919 | Val Loss: 0.119023 | Train MSE (orig): 9.279984 | Val MSE (orig): 8.823726
Epoch [8/50] | Train Loss: 0.116445 | Val Loss: 0.115885 | Train MSE (orig): 8.582375 | Val MSE (orig): 8.590841
Epoch [9/50] | Train Loss: 0.108608 | Val Loss: 0.102402 | Train MSE (orig): 8.003960 |

In [18]:
# Evaluate on test set
targets_all, preds_all = evaluate_baseline(model, test_dl, target_scaler, device)

# Flatten for metrics (handles multi-horizon)
if preds_all.ndim > 1:
    preds_all = preds_all.reshape(-1)
    targets_all = targets_all.reshape(-1)

# Compute metrics
mse, rmse, mae, r2 = compute_metrics(targets_all, preds_all)

print("\nGRU Baseline Model Performance:")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")



GRU Baseline Model Performance:
MSE:  1.8408
RMSE: 1.3568
MAE:  1.0212
R²:   0.9746


In [19]:
class CNNLSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, horizon, dropout=0.1, device=device):
        super(CNNLSTMRegressor, self).__init__()
        self.device = device

        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(32, hidden_dim, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, horizon)
        self.to(self.device)

    def forward(self, x):
        x = x.to(self.device)
        x = x.transpose(1, 2)           # [B, F, T] for Conv1D
        x = self.relu(self.conv1(x))    # [B, 32, T]
        x = x.transpose(1, 2)           # [B, T, 32] for LSTM
        _, (h_n, _) = self.lstm(x)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        h = self.dropout(h)
        out = self.fc(h)
        return out


In [20]:
# Example dimensions
input_dim = 6       # number of features
hidden_dim = 64
horizon = 5

cnn_model = CNNLSTMRegressor(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    horizon=horizon,
    dropout=0.1
).to(device)


In [21]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-5)

train_losses, val_losses, train_mse_orig, val_mse_orig = train_baseline_model(
    model=cnn_model,
    train_loader=train_dl,
    val_loader=val_dl,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=50,
    target_scaler=target_scaler,
    device=device
)


Epoch [1/50] | Train Loss: 0.569806 | Val Loss: 0.312037 | Train MSE (orig): 42.000031 | Val MSE (orig): 22.914782
Epoch [2/50] | Train Loss: 0.275469 | Val Loss: 0.251060 | Train MSE (orig): 20.302336 | Val MSE (orig): 18.547035
Epoch [3/50] | Train Loss: 0.247322 | Val Loss: 0.205735 | Train MSE (orig): 18.219320 | Val MSE (orig): 15.251594
Epoch [4/50] | Train Loss: 0.191040 | Val Loss: 0.184866 | Train MSE (orig): 14.080005 | Val MSE (orig): 13.709979
Epoch [5/50] | Train Loss: 0.165904 | Val Loss: 0.152869 | Train MSE (orig): 12.228380 | Val MSE (orig): 11.338286
Epoch [6/50] | Train Loss: 0.158709 | Val Loss: 0.150961 | Train MSE (orig): 11.698503 | Val MSE (orig): 11.215217
Epoch [7/50] | Train Loss: 0.140283 | Val Loss: 0.139442 | Train MSE (orig): 10.339899 | Val MSE (orig): 10.342854
Epoch [8/50] | Train Loss: 0.132998 | Val Loss: 0.116979 | Train MSE (orig): 9.803964 | Val MSE (orig): 8.668044
Epoch [9/50] | Train Loss: 0.117027 | Val Loss: 0.113107 | Train MSE (orig): 8.624

In [22]:
# Evaluate on test set
targets_all, preds_all = evaluate_baseline(cnn_model, test_dl, target_scaler, device)

# Flatten for metrics (handles multi-horizon)
if preds_all.ndim > 1:
    preds_all = preds_all.reshape(-1)
    targets_all = targets_all.reshape(-1)

# Compute metrics
mse, rmse, mae, r2 = compute_metrics(targets_all, preds_all)

print("\nCNNLSTM Baseline Model Performance:")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")



CNNLSTM Baseline Model Performance:
MSE:  2.0149
RMSE: 1.4195
MAE:  1.0488
R²:   0.9722


In [23]:
class TransformerRegressor(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, horizon, dropout=0.1, device='cpu'):
        super(TransformerRegressor, self).__init__()
        self.device = device
        self.input_proj = nn.Linear(input_dim, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=num_heads, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(model_dim, horizon)
        self.to(self.device)

    def forward(self, x):
        x = x.to(self.device)
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)    # global average pooling
        out = self.fc(x)
        return out


In [24]:
import torch
import torch.nn as nn

# Example dimensions
input_dim = 6       # number of features
model_dim = 64      # embedding dimension inside transformer
num_heads = 8       # number of attention heads
horizon = 5
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Instantiate the model
tranmodel = TransformerRegressor(
    input_dim=input_dim,
    model_dim=model_dim,
    num_heads=num_heads,
    num_layers=2,
    horizon=horizon,
    dropout=0.1,
    device=device
).to(device)

# Test with dummy input
x_dummy = torch.randn(16, 10, input_dim).to(device)  # batch_size=16, seq_len=10
y_pred = tranmodel(x_dummy)
print(y_pred.shape)  # Expected: [16, horizon]


torch.Size([16, 5])


In [25]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(tranmodel.parameters(), lr=1e-3, weight_decay=1e-5)

train_losses, val_losses, train_mse_orig, val_mse_orig = train_baseline_model(
    model=tranmodel,
    train_loader=train_dl,
    val_loader=val_dl,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=50,
    target_scaler=target_scaler,
    device=device
)


Epoch [1/50] | Train Loss: 0.399615 | Val Loss: 0.209943 | Train MSE (orig): 29.456629 | Val MSE (orig): 15.577332
Epoch [2/50] | Train Loss: 0.200198 | Val Loss: 0.167419 | Train MSE (orig): 14.756118 | Val MSE (orig): 12.426858
Epoch [3/50] | Train Loss: 0.161638 | Val Loss: 0.144074 | Train MSE (orig): 11.913496 | Val MSE (orig): 10.686645
Epoch [4/50] | Train Loss: 0.141548 | Val Loss: 0.137331 | Train MSE (orig): 10.432548 | Val MSE (orig): 10.187062
Epoch [5/50] | Train Loss: 0.145651 | Val Loss: 0.128478 | Train MSE (orig): 10.731886 | Val MSE (orig): 9.517738
Epoch [6/50] | Train Loss: 0.117085 | Val Loss: 0.119058 | Train MSE (orig): 8.628453 | Val MSE (orig): 8.792195
Epoch [7/50] | Train Loss: 0.105933 | Val Loss: 0.122766 | Train MSE (orig): 7.807156 | Val MSE (orig): 9.083676
Epoch [8/50] | Train Loss: 0.096499 | Val Loss: 0.097034 | Train MSE (orig): 7.112595 | Val MSE (orig): 7.185971
Epoch [9/50] | Train Loss: 0.087586 | Val Loss: 0.107149 | Train MSE (orig): 6.454162 |

In [26]:
# Evaluate on test set
targets_all, preds_all = evaluate_baseline(tranmodel, test_dl, target_scaler, device)

# Flatten for metrics (handles multi-horizon)
if preds_all.ndim > 1:
    preds_all = preds_all.reshape(-1)
    targets_all = targets_all.reshape(-1)

# Compute metrics
mse, rmse, mae, r2 = compute_metrics(targets_all, preds_all)

print("\nTransformer Baseline Model Performance:")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")



Transformer Baseline Model Performance:
MSE:  2.8685
RMSE: 1.6937
MAE:  1.2662
R²:   0.9604


In [27]:
import pandas as pd

# List of models and their names
models = {
    "Baseline": baseline_model,
    "GRU": model,
    "CNNLSTM": cnn_model,
    "Transformer": tranmodel
}

# Store metrics
metrics_list = []

for name, mdl in models.items():
    targets_all, preds_all = evaluate_baseline(mdl, test_dl, target_scaler, device)
    
    # Flatten if multi-horizon
    if preds_all.ndim > 1:
        preds_all = preds_all.reshape(-1)
        targets_all = targets_all.reshape(-1)
    
    mse, rmse, mae, r2 = compute_metrics(targets_all, preds_all)
    
    metrics_list.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    })

# Convert to DataFrame for nice tabular display
metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df.set_index("Model")

print("\n📊 Test Set Performance Comparison:")
print(metrics_df)



📊 Test Set Performance Comparison:
                  MSE      RMSE       MAE        R²
Model                                              
Baseline     3.795153  1.948115  1.322788  0.947667
GRU          1.840848  1.356779  1.021204  0.974616
CNNLSTM      2.014936  1.419484  1.048753  0.972215
Transformer  2.868458  1.693652  1.266175  0.960446
